In [2]:
from semanticscholar import SemanticScholar
from crossref.restful import Works
from itertools import product
import pandas as pd
import json
import dblp
import requests

In [2]:
crossref_work = Works()
sch = SemanticScholar()

In [3]:
# search params
primary_keywords = [
    "Generative AI",
    "LLM",
    "LM",
    "GenAI",
    "large language model",
    "language model",
    "codex",
    "gpt",
    "gpt-3",
    "gpt-4",
]
secondary_keywords = [
    "overreliance",
    "misinformation",
    "accessbility",
    "privacy",
    "enviromental",
    "explainability",
    "trustworthy",
    "responsible",
]
others = ["mitigation", "ethics", "societal", "social", "ethical"]


sch_year = "2019-"
crossref_year = "2019"
years = ["2019", "2020", "2021", "2022", "2023"]
dblp_formatted_years = " " + "|".join(f"{year}" for year in years)

all_combinations = list(product(primary_keywords, secondary_keywords, others))
all_combinations_no_tertiary = list(product(primary_keywords, secondary_keywords))

In [4]:
# Prepare DataFrame to store the results
columns = [
    'PaperTitle',
    'ID',    
    'SearchString',
    'SearchedFrom',
]
search_results_df = pd.DataFrame(columns=columns)

In [5]:
# SemanticScholar fields
fields = [
    "title",
    "externalIds",
    "paperId",
    "url"
]

In [6]:
import time
search_results_df = pd.DataFrame(columns=columns)
try:
    for search_string in all_combinations:
        
        print(f"------------------------------- searching for {search_string} ({all_combinations.index(search_string)}/{len(all_combinations)}) ------------------------------")
        
        # Semantic Scholar
         # send http request to api
        api_url = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
        headers = {"Content-Type": "application/json", "x-api-key": "X48LIBLqr86ouHlnMYd3z052sgEm3Nd2wMORPzu5"}
        sch_search_string = ' + '.join(search_string)
        params = {"query": sch_search_string, "year": sch_year, "fields": ",".join(fields)}
        response = requests.get(api_url, headers=headers, params=params)
        while response.status_code != 200:
            print(f"Semantic Scholar response status code: {response.status_code}, waiting 30 seconds...")
            time.sleep(30)
            response = requests.get(api_url, headers=headers, params=params)
        response_json = response.json()
        sch_results = response_json["data"]
        print(f">>> Semantic scholar total: {response_json['total']}")
        # Add results to DataFrame
        sch_count = 0
        if sch_results != 0:
            for result in sch_results:
                sch_count += 1
                if result['externalIds'].get('DOI') is None:
                    if result['externalIds'].get('ArXiv') is None:
                        paper_id = "paperid:" + result.get("paperId")
                    else:
                        paper_id = "DOI:10.48550/arXiv." + result['externalIds'].get('ArXiv')
                else:
                    paper_id = f"DOI:{result['externalIds'].get('DOI')}"
                new_paper = {
                    'PaperTitle': result['title'],
                    'ID': paper_id,
                    'SearchString': sch_search_string,
                    'SearchedFrom': 'Semantic Scholar'
                }
                print(f"{sch_count}. sch process paper: ", new_paper)
                search_results_df = pd.concat([search_results_df, pd.DataFrame([new_paper])], ignore_index=True) 
        
        # # # Crossref
        # # TODO: too many result
        # crossref_search_string = ' '.join(search_string)
        # cr_search_results = crossref_work.query(crossref_search_string).filter(from_online_pub_date=crossref_year)
        # print(f"Crossref total: {cr_search_results.count()}")
        # crossref_count = 0
        # for cr_result in cr_search_results:
        #     crossref_count += 1
        #     new_paper = {
        #         'PaperTitle': cr_result.get('title'),
        #         'DOI': cr_result.get('DOI'),
        #         'SearchString': crossref_search_string,
        #         'SearchedFrom': 'Crossref'
        #     }
        #     print(f"{crossref_count}. Crossref process paper: ", new_paper)
        #     search_results_df = pd.concat([search_results_df, pd.DataFrame([new_paper])], ignore_index=True) 

        # # search in dblp
        # dblp_search_string = ' '.join(search_string)
        # dblp_search_results = dblp.search(dblp_search_string + dblp_formatted_years)
        # if dblp_search_results is None:
        #     print(f"\n>>> DBLP total: 0")
        # else:
        #     print(f"\n>>> DBLP total: {len(dblp_search_results)}")
        #     dblp_count = 0
        #     for key, result in dblp_search_results.items():
        #         dblp_count += 1
        #         new_paper = {
        #             'PaperTitle': result.get('title'),
        #             'DOI': result.get('doi'),
        #             'SearchString': dblp_search_string,
        #             'SearchedFrom': 'DBLP'
        #         }
        #         print(f"{dblp_count}. DBLP process paper: ", new_paper)
        #         search_results_df = pd.concat([search_results_df, pd.DataFrame([new_paper])], ignore_index=True)
        print(f"^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for {search_string} ({all_combinations.index(search_string)}/{len(all_combinations)}) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n\n")
    search_results_df.to_csv('data/search-results-sch.csv', index=False)
    print("XXXXXXXXXXXXXXXXXXXXXXXXXXXX search ends XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
except Exception as e:
    search_results_df.to_csv('data/search-results-sch.csv', index=False)
    print(f"An error occurred: {e.with_traceback()}")
    print("XXXXXXXXXXXXXXXXXXXXXXXXXXXX search ends XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")


------------------------------- searching for ('Generative AI', 'overreliance', 'mitigation') (0/400) ------------------------------
>>> Semantic scholar total: 2
1. sch process paper:  {'PaperTitle': 'Vox Populi, Vox ChatGPT: Large Language Models, Education and Democracy', 'ID': 'DOI:10.48550/arXiv.2311.06207', 'SearchString': 'Generative AI + overreliance + mitigation', 'SearchedFrom': 'Semantic Scholar'}
2. sch process paper:  {'PaperTitle': 'On the Philosophy of Unsupervised Learning', 'ID': 'DOI:10.1007/s13347-023-00635-6', 'SearchString': 'Generative AI + overreliance + mitigation', 'SearchedFrom': 'Semantic Scholar'}
^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for ('Generative AI', 'overreliance', 'mitigation') (0/400) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


------------------------------- searching for ('Generative AI', 'overreliance', 'ethics') (1/400) ------------------------------
>>> Semantic scholar total: 0
^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for ('Generative AI', 'overre

In [7]:
# dblp WITH tertiary level keywords:
search_results_df = pd.DataFrame(columns=columns)
try:
    for search_string in all_combinations:
        
        print(f"------------------------------- searching for {search_string} ({all_combinations.index(search_string)}/{len(all_combinations)}) ------------------------------")
        # search in dblp
        dblp_search_string = ' '.join(search_string)
        dblp_search_results = dblp.search(dblp_search_string + dblp_formatted_years)
        if dblp_search_results is None:
            print(f">>> DBLP total: 0")
        else:
            print(f">>> DBLP total: {len(dblp_search_results)}")
            dblp_count = 0
            for key, result in dblp_search_results.items():
                dblp_count += 1
                new_paper = {
                    'PaperTitle': result.get('info').get('title'),
                    'ID': "DOI:" + result.get('info').get('doi'),
                    'SearchString': dblp_search_string,
                    'SearchedFrom': 'DBLP'
                }
                print(f"{dblp_count}. DBLP process paper: ", new_paper)
                search_results_df = pd.concat([search_results_df, pd.DataFrame([new_paper])], ignore_index=True)
        print(f"^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for {search_string} ({all_combinations.index(search_string)}/{len(all_combinations)}) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n\n")
    search_results_df.to_csv('data/search-results-dblp.csv', index=False)
    print("XXXXXXXXXXXXXXXXXXXXXXXXXXXX search ends XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
except Exception as e:
    search_results_df.to_csv('data/search-results-dblp.csv', index=False)
    print(f"An error occurred: {e.with_traceback()}")
    print("XXXXXXXXXXXXXXXXXXXXXXXXXXXX search ends XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")


------------------------------- searching for ('Generative AI', 'overreliance', 'mitigation') (0/400) ------------------------------
>>> DBLP total: 0
^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for ('Generative AI', 'overreliance', 'mitigation') (0/400) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


------------------------------- searching for ('Generative AI', 'overreliance', 'ethics') (1/400) ------------------------------
>>> DBLP total: 0
^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for ('Generative AI', 'overreliance', 'ethics') (1/400) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


------------------------------- searching for ('Generative AI', 'overreliance', 'societal') (2/400) ------------------------------
>>> DBLP total: 0
^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for ('Generative AI', 'overreliance', 'societal') (2/400) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


------------------------------- searching for ('Generative AI', 'overreliance', 'social') (3/400) ------------------------------
>>> DBLP total: 0
^

In [11]:
# dblp WITHOUT tertiary level keywords:
search_results_df = pd.DataFrame(columns=columns)
try:
    for search_string in all_combinations_no_tertiary:
        
        print(f"------------------------------- searching for {search_string} ({all_combinations_no_tertiary.index(search_string)}/{len(all_combinations_no_tertiary)}) ------------------------------")
        # search in dblp
        dblp_search_string = ' '.join(search_string)
        dblp_search_results = dblp.search(dblp_search_string + dblp_formatted_years)
        if dblp_search_results is None:
            print(f">>> DBLP total: 0")
        else:
            print(f">>> DBLP total: {len(dblp_search_results)}")
            dblp_count = 0
            for key, result in dblp_search_results.items():
                dblp_count += 1
                if result.get('info').get('doi') is None:
                    paper_id = "url:" + result.get("info").get("url")
                else:
                    paper_id = "DOI:" + result.get('info').get('doi')
                new_paper = {
                    'PaperTitle': result.get('info').get('title'),
                    'ID': paper_id,
                    'SearchString': dblp_search_string,
                    'SearchedFrom': 'DBLP'
                }
                print(f"{dblp_count}. DBLP process paper: ", new_paper)
                search_results_df = pd.concat([search_results_df, pd.DataFrame([new_paper])], ignore_index=True)
        print(f"^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for {search_string} ({all_combinations_no_tertiary.index(search_string)}/{len(all_combinations_no_tertiary)}) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n\n")
    search_results_df.to_csv('data/raw/search-results-dblp-no-tertiary.csv', index=False)
    print("XXXXXXXXXXXXXXXXXXXXXXXXXXXX search ends XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
except Exception as e:
    search_results_df.to_csv('data/raw/search-results-dblp-no-tertiary.csv', index=False)
    print(f"An error occurred: {e.with_traceback()}")
    print("XXXXXXXXXXXXXXXXXXXXXXXXXXXX search ends XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

------------------------------- searching for ('Generative AI', 'overreliance') (0/80) ------------------------------
>>> DBLP total: 0
^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for ('Generative AI', 'overreliance') (0/80) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


------------------------------- searching for ('Generative AI', 'misinformation') (1/80) ------------------------------
>>> DBLP total: 2
1. DBLP process paper:  {'PaperTitle': 'Deepfakes, Misinformation, and Disinformation in the Era of Frontier AI, Generative AI, and Large AI Models.', 'ID': 'DOI:10.48550/ARXIV.2311.17394', 'SearchString': 'Generative AI misinformation', 'SearchedFrom': 'DBLP'}
2. DBLP process paper:  {'PaperTitle': 'Combating Misinformation in the Era of Generative AI Models.', 'ID': 'DOI:10.1145/3581783.3612704', 'SearchString': 'Generative AI misinformation', 'SearchedFrom': 'DBLP'}
^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for ('Generative AI', 'misinformation') (1/80) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


------

In [3]:
# remove duplicates internally
sch_search_results_df = pd.read_csv('data/raw/search-results-sch.csv')
dblp_search_results_df = pd.read_csv('data/raw/search-results-dblp.csv')
dblp_search_results_df_no_tertiary = pd.read_csv('data/raw/search-results-dblp-no-tertiary.csv')

# First, remove duplicates based on the 'ID'
sch_search_results_df = sch_search_results_df.drop_duplicates(subset=['ID'])
dblp_search_results_df = dblp_search_results_df.drop_duplicates(subset=['ID'])
dblp_search_results_df_no_tertiary = dblp_search_results_df_no_tertiary.drop_duplicates(subset=['ID'])

# Next, remove duplicates based on the 'PaperTitle'
sch_search_results_df = sch_search_results_df.drop_duplicates(subset=['PaperTitle'])
dblp_search_results_df = dblp_search_results_df.drop_duplicates(subset=['PaperTitle'])
dblp_search_results_df_no_tertiary = dblp_search_results_df_no_tertiary.drop_duplicates(subset=['PaperTitle'])


sch_search_results_df.to_csv('data/cleaned/search-results-sch-cleaned.csv', index=False)
dblp_search_results_df.to_csv('data/cleaned/search-results-dblp-cleaned.csv', index=False)
dblp_search_results_df_no_tertiary.to_csv('data/cleaned/search-results-dblp-no-tertiary-cleaned.csv', index=False)


# Testing stuff

In [ ]:
search_string = 'Ormco Unveils SymetriTM Clear ceramic twin bracket system'
sch_test = sch.search_paper(search_string)

In [ ]:
search_string = 'Generative AI Social Impact education'
cr_url = crossref_work.query(search_string).filter(from_online_pub_date=crossref_year, 
                                                              type="journal-article").filter(type="journal-article").url
cr_search_results = crossref_work.query(search_string).filter(from_online_pub_date=crossref_year, 
                                                              type="journal-article dd").filter(type="journal-article").count()

In [ ]:
search_string = 'large language model misinformation' + " 2019|2020|2021|2022|2023"
dblp_test = dblp.search(search_string)
for key, result in dblp_test.items():
                new_paper = {
                    'PaperTitle': result.get('info').get('title'),
                    'DOI': result.get('info').get('doi'),
                    'SearchedFrom': 'DBLP'
                }
                print(f"DBLP process paper: ", new_paper)

In [ ]:
api_url = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
headers = {"Content-Type": "application/json"}
params = {"query": "Generative AI + misinformation + social", "year": sch_year, "fields": ",".join(fields)}
response = requests.get(api_url, headers=headers, params=params)
response_json = response.json()